In [1]:
# ============================================================
# Create Sample Input File
# ============================================================

import pandas as pd
import os

os.makedirs("../samples", exist_ok=True)

DATA_PATH = "../data/train.csv"

data = pd.read_csv(DATA_PATH)

TARGET = "diagnosed_diabetes"

sample_input = data.drop(columns=[TARGET, "id"]).sample(
    n=10,
    random_state=42
)

sample_input.to_csv(
    "../samples/sample_input.csv",
    index=False
)

print(sample_input)

print("\n✓ sample_input.csv created successfully.")

        age  alcohol_consumption_per_week  physical_activity_minutes_per_week  \
637949   60                             2                                 144   
656465   50                             2                                  48   
337739   34                             1                                 123   
230222   52                             2                                  26   
82039    39                             3                                  70   
333655   68                             3                                  80   
668993   40                             1                                  95   
394818   67                             2                                  42   
54454    26                             1                                  12   
211991   49                             1                                 381   

        diet_score  sleep_hours_per_day  screen_time_hours_per_day   bmi  \
637949         4.3              

In [2]:
# ============================================================
# Load Production Models
# ============================================================

import joblib

preprocessor = joblib.load("../models/preprocessor.pkl")

best_xgb = joblib.load("../models/xgb.pkl")

best_lgb = joblib.load("../models/lightgbm.pkl")

best_mlp = joblib.load("../models/mlp.pkl")

calibrated_model = joblib.load("../models/calibrated.pkl")

best_threshold = joblib.load("../models/threshold.pkl")

target_labels = joblib.load("../models/target_labels.pkl")

print("✓ Models loaded successfully.")

✓ Models loaded successfully.


In [3]:
# ============================================================
# Read Sample Input
# ============================================================

sample_input = pd.read_csv("../samples/sample_input.csv")

print(sample_input.head())

   age  alcohol_consumption_per_week  physical_activity_minutes_per_week  \
0   60                             2                                 144   
1   50                             2                                  48   
2   34                             1                                 123   
3   52                             2                                  26   
4   39                             3                                  70   

   diet_score  sleep_hours_per_day  screen_time_hours_per_day   bmi  \
0         4.3                  7.3                        5.5  23.1   
1         4.7                  7.2                        9.1  26.6   
2         3.4                  7.0                        7.1  24.1   
3         6.1                  7.3                        5.0  22.9   
4         4.9                  8.8                        7.8  29.3   

   waist_to_hip_ratio  systolic_bp  diastolic_bp  ...  triglycerides  gender  \
0                0.83          100  

In [4]:
# ============================================================
# Generate Predictions
# ============================================================

sample_processed = preprocessor.transform(sample_input)

xgb_prob = best_xgb.predict_proba(sample_processed)[:, 1]

lgb_prob = best_lgb.predict_proba(sample_processed)[:, 1]

mlp_prob = best_mlp.predict_proba(sample_processed)[:, 1]

meta_features = pd.DataFrame({

    "XGB": xgb_prob,
    "LGBM": lgb_prob,
    "MLP": mlp_prob

})

probabilities = calibrated_model.predict_proba(
    meta_features
)[:, 1]

predictions = (
    probabilities >= best_threshold
).astype(int)

classes = [
    target_labels[p]
    for p in predictions
]

d:\Projects\Diabetes-Risk-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
# ============================================================
# Save Sample Output
# ============================================================

sample_output = sample_input.copy()

sample_output["Probability"] = probabilities
sample_output["Threshold"] = best_threshold
sample_output["Prediction"] = predictions
sample_output["Class"] = classes

sample_output.to_csv(
    "../samples/sample_output.csv",
    index=False
)

print(sample_output)

print("\n✓ sample_output.csv created successfully.")

   age  alcohol_consumption_per_week  physical_activity_minutes_per_week  \
0   60                             2                                 144   
1   50                             2                                  48   
2   34                             1                                 123   
3   52                             2                                  26   
4   39                             3                                  70   
5   68                             3                                  80   
6   40                             1                                  95   
7   67                             2                                  42   
8   26                             1                                  12   
9   49                             1                                 381   

   diet_score  sleep_hours_per_day  screen_time_hours_per_day   bmi  \
0         4.3                  7.3                        5.5  23.1   
1         4.7        